[env: optonpx]

# 1.5 Detect bad channels

A reference for the `bad_channels` of each probe in the session metadata, to run **before** notebook
2.1. The bad channels are used by the common-mode noise detection in 2.1, by the channel map (2.2)
and by Kilosort (2.3), so they should be right before 2.1 runs.

This notebook only reads the raw recordings and **writes nothing**: it prints a suggested list and
you decide what goes into the metadata yaml.

## What is checked

Every probe of every session in the metadata (behavior, optotagging) is checked on its own. A channel
that is bad in any session is suggested, because 2.1 concatenates the sessions and Kilosort sorts
them together.

1. **SpikeInterface `detect_bad_channels(method="coherence+psd")`** (the IBL method). It compares
   every channel with the median of the probe, after a 300 Hz high-pass, on 100 random 0.3 s chunks
   (fixed seed, so every run gives the same labels):
   - `dead`: much less similar to the probe median than its neighbours → **suggested**;
   - `noise`: much more similar than its neighbours, or high power near the Nyquist frequency → **suggested**;
   - `out`: a contiguous block at the top of a shank that is not similar to the rest (outside the
     brain) → **only shown**, not suggested.

   The method orders the channels by depth, so it is run on each shank separately.
2. **Saturated (at the rail)**: the fraction of samples at the largest value the probe produces
   (± the same value; ±31331 or ±31363 here). A channel that is at the rail for a large part of the
   time (`rail_fraction_threshold`, 0.1 %) carries no signal while it is there, and every time it
   comes off the rail (e.g. pulled by the laser artifact) it makes a large step → **suggested**.
   Channels that are stuck at the rail all the time are also caught here.
3. **Flat**: the median absolute first difference is 0, i.e. the value does not change from one sample
   to the next for most of the time → **suggested**.
4. **Large DC offset**: the median of the raw signal is more than `offset_fraction_threshold` (50 %)
   of the rail. Such a channel is close to saturating and may clip when the signal drifts → **only
   shown**; look at its rail fraction.

Criteria 2–4 use 1000 evenly spaced 0.1 s chunks of the raw (unfiltered) data of each session (100 s
in all), so they need no random seed.

## How to use the result

- For each problem type, the channels are listed per session.
- "suggest adding" are flagged channels that are not in the metadata yet; "in the metadata, not
  flagged" are channels you put there for another reason (or that recovered); keep them unless you
  know they are fine.
- The last line of each probe is a `bad_channels: [...]` line (metadata ∪ suggestions) that can be
  pasted into the yaml after checking it.

In [ ]:
from pathlib import Path

import yaml

# ========== Specify the session to check ==========
session_name = "IM-1979_2026-09-17"
base_dir_raw = Path(r"D:/")

rail_fraction_threshold = 0.001     # at the rail for more than 0.1 % of the samples -> suggested
offset_fraction_threshold = 0.5     # |median| above 50 % of the rail -> shown
n_chunks, chunk_s = 1000, 0.1       # evenly spaced raw chunks for the rail / flat / offset metrics
si_seed = 0                         # random chunks of detect_bad_channels
# ==================================================

metadata_path = Path.cwd().parent / "metadata" / f"{session_name}.yaml"
config = yaml.safe_load(metadata_path.read_text(encoding="utf-8"))
probe_names = config["streams"]["probes"]
session_roles = [role for role in config["sessions"] if role != "processed"]
print(metadata_path)
print(f"probes {probe_names}, sessions {session_roles}")

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append("..")
import spikeinterface.full as si
from spikeinterface.preprocessing import detect_bad_channels

from src import oe_parse_folders, oe_parse_params

print("spikeinterface", si.__version__)


def match_one(items, probe_name, name_of):
    """The single item whose name ends with probe_name, case-insensitively ('probeA' vs 'ProbeA')."""
    matches = [item for item in items if str(name_of(item)).split(".")[-1].lower() == probe_name.lower()]
    if len(matches) != 1:
        raise ValueError(f"{probe_name!r}: expected 1 match, got {[name_of(item) for item in items]}")
    return matches[0]


streams = {}   # (probe_name, role) -> {dat, n_chan, fs, params}
for role in session_roles:
    folder = base_dir_raw / config["animal"] / "ephys" / config["sessions"][role]
    _, oe_paths = oe_parse_folders(folder)
    probe_params_list, _ = oe_parse_params(oe_paths["xml"], oe_paths["oebin"])
    for probe_name in probe_names:
        params = match_one(probe_params_list, probe_name, lambda p: p["stream_name"])
        stream_dir = match_one(oe_paths["ephys_streams"], probe_name, lambda p: p.name)
        streams[(probe_name, role)] = {"dat": stream_dir / "continuous.dat", "n_chan": int(params["channel_count"]),
                                       "fs": float(params["sample_rate"]), "params": params}
        print(f"{role:12s} {probe_name}: {stream_dir}")

In [ ]:
# --- per probe x session: raw metrics on evenly spaced chunks + SpikeInterface labels per shank ---
rows = []
for (probe_name, role), stream in streams.items():
    n_chan, fs, params = stream["n_chan"], stream["fs"], stream["params"]
    raw = np.memmap(stream["dat"], dtype="int16", mode="r").reshape(-1, n_chan)   # read-only

    # raw metrics
    chunk = int(chunk_s * fs)
    starts = np.linspace(0, raw.shape[0] - chunk, n_chunks).astype(int)
    data = np.stack([raw[start:start + chunk] for start in starts])               # chunk x time x channel
    rail = int(np.abs(data).max())                                                 # the largest value of the probe
    rail_fraction = (np.abs(data) >= rail).mean(axis=(0, 1))
    step_mad = np.median(np.abs(np.diff(data.astype(np.int32), axis=1)), axis=(0, 1))   # within chunks only
    offset_fraction = np.abs(np.median(data, axis=(0, 1))) / rail

    # SpikeInterface coherence+psd, one shank at a time (the method orders the channels by depth)
    recording = si.read_binary(stream["dat"], sampling_frequency=fs, dtype="int16", num_channels=n_chan)
    recording.set_channel_gains(float(params["channel_bit_volts"]))               # µV per bit
    recording.set_channel_offsets(0.0)
    recording.set_channel_locations(np.array([[params["channel_xpos"][ch], params["channel_ypos"][ch]]
                                              for ch in range(n_chan)], dtype=float))
    shank_of = np.array([params["channel_shank"][ch] for ch in range(n_chan)])
    si_label = np.empty(n_chan, dtype=object)
    for shank in np.unique(shank_of):
        channels = np.flatnonzero(shank_of == shank)
        _, labels = detect_bad_channels(recording.select_channels(recording.channel_ids[channels]),
                                        method="coherence+psd", seed=si_seed)
        si_label[channels] = labels

    for ch in range(n_chan):
        rows.append({"probe": probe_name, "session": role, "channel": ch, "shank": int(shank_of[ch]),
                     "y_um": float(params["channel_ypos"][ch]), "si_label": si_label[ch],
                     "rail_fraction": float(rail_fraction[ch]), "step_mad": float(step_mad[ch]),
                     "offset_fraction": float(offset_fraction[ch]), "rail": rail})
    print(f"{probe_name} {role:12s}: rail ±{rail}, SI labels "
          + ", ".join(f"{label} {int((si_label == label).sum())}" for label in ("dead", "noise", "out")))

metrics = pd.DataFrame(rows)
metrics["suggest_reason"] = ""
metrics.loc[metrics["si_label"].isin(["dead", "noise"]), "suggest_reason"] += "SI " + metrics["si_label"] + "; "
metrics.loc[metrics["rail_fraction"] >= rail_fraction_threshold, "suggest_reason"] += "rail; "
metrics.loc[metrics["step_mad"] == 0, "suggest_reason"] += "flat; "
metrics["show_reason"] = ""
metrics.loc[metrics["si_label"] == "out", "show_reason"] += "SI out; "
metrics.loc[metrics["offset_fraction"] >= offset_fraction_threshold, "show_reason"] += "offset; "

In [ ]:
# --- per probe: the channels of every problem type, per session; then the suggestion ---
suggested_types = {   # problem type -> test on the metrics table
    "SI dead":   lambda m: m["si_label"] == "dead",
    "SI noise":  lambda m: m["si_label"] == "noise",
    "saturated": lambda m: m["rail_fraction"] >= rail_fraction_threshold,
    "flat":      lambda m: m["step_mad"] == 0,
}
shown_types = {
    "SI out":          lambda m: m["si_label"] == "out",
    "large DC offset": lambda m: m["offset_fraction"] >= offset_fraction_threshold,
}


def print_type(probe_metrics, name, test):
    """Print the channels of one problem type per session; return them all."""
    found = set()
    for role in session_roles:
        session_metrics = probe_metrics[probe_metrics["session"] == role]
        channels = sorted(session_metrics.loc[test(session_metrics), "channel"])
        print(f"  {name:16s} {role:12s}: {channels}")
        found |= set(channels)
    return found


for probe_name in probe_names:
    probe_metrics = metrics[metrics["probe"] == probe_name]
    in_metadata = set(config[probe_name]["kilosort_params"]["bad_channels"])
    print(f"\n========== {probe_name} ({config[probe_name]['target']['region']}) ==========")
    print(f"bad_channels in the metadata: {sorted(in_metadata)}\n")

    print("suggested:")
    flagged = set()
    for name, test in suggested_types.items():
        flagged |= print_type(probe_metrics, name, test)
    print("only shown, not suggested:")
    for name, test in shown_types.items():
        print_type(probe_metrics, name, test)

    print(f"\nsuggest adding              : {sorted(flagged - in_metadata)}")
    print(f"in the metadata, not flagged: {sorted(in_metadata - flagged)}")
    print(f"{probe_name}:  bad_channels: {sorted(in_metadata | flagged)}")

In [ ]:
# --- per probe: every metric per channel and session, with the thresholds ---
session_colors = {role: color for role, color in zip(session_roles, ["tab:blue", "tab:orange", "tab:green", "tab:red"])}
label_level = {"good": 0, "out": 1, "dead": 2, "noise": 3}

for probe_name in probe_names:
    probe_metrics = metrics[metrics["probe"] == probe_name]
    in_metadata = config[probe_name]["kilosort_params"]["bad_channels"]
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), dpi=150, sharex=True)
    for role in session_roles:
        session_metrics = probe_metrics[probe_metrics["session"] == role]
        style = {"color": session_colors[role], "s": 8, "label": role}
        not_good = session_metrics[session_metrics["si_label"] != "good"]
        axes[0].scatter(not_good["channel"], not_good["si_label"].map(label_level)
                        + (0.15 if role == session_roles[-1] else -0.15), **style)
        axes[1].scatter(session_metrics["channel"], np.maximum(session_metrics["rail_fraction"], 1e-6), **style)
        axes[2].scatter(session_metrics["channel"], np.maximum(session_metrics["step_mad"], 0.1), **style)
        axes[3].scatter(session_metrics["channel"], session_metrics["offset_fraction"], **style)
    axes[0].set(yticks=[1, 2, 3], yticklabels=["out", "dead", "noise"], ylim=(0.5, 3.5),
                title=f"{probe_name}: SpikeInterface label (only channels that are not 'good')")
    axes[1].axhline(rail_fraction_threshold, color="k", lw=0.8, ls="--")
    axes[1].set(yscale="log", ylabel="fraction at the rail", title="saturation (1e-6 = never)")
    axes[2].set(yscale="log", ylabel="median |first diff| (bits)", title="sample-to-sample variation (0.1 = flat)")
    axes[3].axhline(offset_fraction_threshold, color="k", lw=0.8, ls="--")
    axes[3].set(ylabel="|median| / rail", xlabel="channel", title="DC offset")
    for ax in axes:
        for ch in in_metadata:
            ax.axvline(ch, color="0.8", lw=0.6, zorder=0)
    axes[1].legend(fontsize=8, loc="upper right")
    fig.suptitle(f"{session_name} {probe_name} — grey lines: bad_channels in the metadata")
    fig.tight_layout()
    plt.show()